# WooCommerce CopyAndPay vs Nochex CopyPay On-Page
Set up a reproducible workflow to fetch the Nochex reference plugin, inspect its gateway class, and capture the insights needed for our comparison.

## 1. Setup Repository Access
Download the latest tree for `nochexdevteam/woocommerce-copypay-on-page`, cache the raw files locally, and list the main plugin assets so later cells can reference them.

In [2]:
import json
from pathlib import Path
from textwrap import indent
import requests
import pandas as pd

REPO = "nochexdevteam/woocommerce-copypay-on-page"
RAW_BASE_TEMPLATE = "https://raw.githubusercontent.com/{repo}/{branch}"
TREE_URL_TEMPLATE = "https://api.github.com/repos/{repo}/git/trees/{branch}?recursive=1"
CACHE_DIR = Path("analysis/.cache/nochex-copypay")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "ncx-cp-api-comparison",
    "Accept": "application/vnd.github+json"
})


def fetch_json(url: str):
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()


repo_meta = fetch_json(f"https://api.github.com/repos/{REPO}")
BRANCH = repo_meta.get("default_branch", "main")
RAW_BASE = RAW_BASE_TEMPLATE.format(repo=REPO, branch=BRANCH)
TREE_URL = TREE_URL_TEMPLATE.format(repo=REPO, branch=BRANCH)

print(f"Using branch: {BRANCH}")

tree_payload = fetch_json(TREE_URL)
all_files = [node for node in tree_payload.get("tree", []) if node["type"] == "blob"]
plugin_files = [node for node in all_files if node["path"].endswith((".php", ".js", ".css"))]
summary_df = pd.DataFrame(plugin_files)
display(summary_df.sort_values("path").reset_index(drop=True).head(30))
print(f"Discovered {len(plugin_files)} plugin-related files out of {len(all_files)} total files.")

Using branch: main


,path,mode,type,sha,size,url
0,nochexapi/admin/class-nochexapi-admin.php,100644,blob,124ecec60045fdacd208b9d640b6c25e85989f54,1283,https://api.github.com/repos/NochexDevTeam/Woo...
1,nochexapi/admin/css/nochexapi-admin.css,100644,blob,97734be6f5d174590ab4677359a2742a563c96a2,3747,https://api.github.com/repos/NochexDevTeam/Woo...
2,nochexapi/admin/index.php,100644,blob,d88d0dc8ec095a7582835d05fb96f363f5087257,27,https://api.github.com/repos/NochexDevTeam/Woo...
3,nochexapi/admin/js/nochexapi-admin.js,100644,blob,0d02b0ed81053bdd8bdd8850d5e387e4a985463f,3505,https://api.github.com/repos/NochexDevTeam/Woo...
4,nochexapi/admin/partials/admin-options.php,100644,blob,b87ce871a19b191681b0d72f8c3d272879a79a72,10855,https://api.github.com/repos/NochexDevTeam/Woo...
5,nochexapi/admin/partials/nochexapi-admin-displ...,100644,blob,22184a819e824dacdd1f90596bb53b88336fb04a,96,https://api.github.com/repos/NochexDevTeam/Woo...
6,nochexapi/assets/css/cards_style.min.css,100644,blob,28bf3c86513d351a64a07b30421d6a503262addd,75,https://api.github.com/repos/NochexDevTeam/Woo...
7,nochexapi/assets/css/cardsv2-style.css,100644,blob,a90fbdd9ca953d7867e4b5d37ce89da4dbf541b7,5616,https://api.github.com/repos/NochexDevTeam/Woo...
8,nochexapi/assets/frames/3d.php,100644,blob,073b8f0e0a76ab2005a82979d2564670b72dc1c6,9,https://api.github.com/repos/NochexDevTeam/Woo...
9,nochexapi/assets/frames/index.php,100644,blob,034a13517c6ea7419bd61ceff03359a8028afba9,11,https://api.github.com/repos/NochexDevTeam/Woo...


Discovered 42 plugin-related files out of 54 total files.


## 2. Inspect Gateway Class Source
Locate the WooCommerce gateway class inside the repository, pull the raw PHP into our cache, and capture the contents of core methods (`__construct`, `init_form_fields`, `process_payment`, callbacks).

In [ ]:
from typing import Dict, List
import re


def possible_gateway_paths(files: List[Dict[str, str]]) -> List[str]:
    hits = []
    for node in files:
        path = node["path"]
        lower = path.lower()
        if "gateway" in lower and lower.endswith(".php"):
            hits.append(path)
    return sorted(hits)


def fetch_raw_file(path: str) -> str:
    url = f"{RAW_BASE}/{path}"
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    target = CACHE_DIR / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(resp.text, encoding="utf-8")
    return resp.text


gateway_candidates = possible_gateway_paths(plugin_files)
print("Gateway candidates:")
print("\n".join(gateway_candidates))

gateway_path = next((p for p in gateway_candidates if p.endswith(".php")), None)
if gateway_path is None:
    raise RuntimeError("Gateway class file not found")

print(f"\nUsing gateway file: {gateway_path}")
gateway_source = fetch_raw_file(gateway_path)
print(f"Loaded {len(gateway_source.splitlines())} lines")

def extract_method(source: str, method: str, window: int = 80) -> str:
    pattern = re.compile(rf"function\s+{method}[^{{]*{{", re.IGNORECASE)
    match = pattern.search(source)
    if not match:
        return f"// method {method} not found"
    start = match.start()
    brace_depth = 0
    idx = match.end()
    while idx < len(source):
        char = source[idx]
        if char == '{':
            brace_depth += 1
        elif char == '}':
            if brace_depth == 0:
                idx += 1
                break
            brace_depth -= 1
        idx += 1
    return source[start:idx]

for method in ["__construct", "init_form_fields", "process_payment", "process_refund", "nochex_callback"]:
    snippet = extract_method(gateway_source, method)
    preview = "\n".join(snippet.splitlines()[:40])
    print("\n" + "=" * 40)
    print(f"Method: {method}")
    print(preview)
    if len(snippet.splitlines()) > 40:
        print("... (truncated)")

## 3. Extract Admin Setting Definitions
Parse `init_form_fields` so every admin option (ID, label, description, type, default) can be compared to our CopyAndPay settings.

In [ ]:
def extract_form_fields_block(source: str) -> str:
    anchor = "this->form_fields"
    start = source.find(anchor)
    if start == -1:
        raise RuntimeError("form_fields assignment not found")
    start = source.find("array", start)
    start = source.find("(", start) + 1
    depth = 1
    idx = start
    while depth and idx < len(source):
        ch = source[idx]
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        idx += 1
    return source[start:idx-1]


def split_field_entries(array_body: str):
    entries = []
    pattern = re.compile(r"'([^']+)'\s*=>\s*array\(")
    pos = 0
    while True:
        match = pattern.search(array_body, pos)
        if not match:
            break
        field_id = match.group(1)
        cursor = match.end() - 1
        depth = 1
        cursor += 1
        start = cursor
        while depth and cursor < len(array_body):
            ch = array_body[cursor]
            if ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
            cursor += 1
        body = array_body[start:cursor-1]
        entries.append((field_id, body.strip()))
        pos = cursor
    return entries


def parse_field_body(body: str) -> Dict[str, str]:
    kv_pattern = re.compile(r"'(?P<key>[^']+)'\s*=>\s*(?P<value>[^,]+)(?:,|$)")
    data = {}
    for match in kv_pattern.finditer(body):
        key = match.group("key")
        raw_value = match.group("value").strip()
        if raw_value.startswith("array"):
            data[key] = raw_value
        elif raw_value.startswith("'") and raw_value.endswith("'"):
            data[key] = raw_value[1:-1]
        elif raw_value.lower() in {"true", "false"}:
            data[key] = raw_value.lower() == "true"
        else:
            data[key] = raw_value
    return data


form_body = extract_form_fields_block(gateway_source)
field_rows = []
for field_id, body in split_field_entries(form_body):
    parsed = parse_field_body(body)
    field_rows.append({
        "id": field_id,
        "title": parsed.get("title"),
        "type": parsed.get("type"),
        "label": parsed.get("label"),
        "description": parsed.get("description"),
        "default": parsed.get("default"),
        "options/raw": parsed.get("options", parsed.get("label"))
    })

fields_df = pd.DataFrame(field_rows)
display(fields_df)

fields_md = fields_df.fillna("").to_markdown(index=False)
print(fields_md)

## 4. Analyze Payment Flow Implementation
Trace how `process_payment` builds CopyPay requests, observes callbacks (`nochex_callback` / `nochex_success`), logs data, and mentions WooCommerce compatibility considerations.

In [ ]:
import textwrap

methods_of_interest = {
    name: extract_method(gateway_source, name)
    for name in [
        "process_payment",
        "receipt_page",
        "nochex_callback",
        "nochex_success",
        "nochex_cancelled",
        "process_refund"
    ]
}

keywords = [
    "wp_remote_post",
    "wp_remote_get",
    "wc_get_order",
    "add_post_meta",
    "update_status",
    "log",
    "wc_add_notice",
    "wp_safe_redirect"
]

summary_rows = []
for name, body in methods_of_interest.items():
    lines = [line.strip() for line in body.splitlines() if line.strip()]
    interesting = [
        line for line in lines
        if any(key in line for key in keywords)
    ]
    summary_rows.append({
        "method": name,
        "line_count": len(lines),
        "key_lines": " \n".join(interesting[:8])
    })

method_df = pd.DataFrame(summary_rows)
display(method_df)

for name, body in methods_of_interest.items():
    print("\n" + "-" * 60)
    print(name.upper())
    print(textwrap.dedent("\n".join(body.splitlines()[:80])))
    if len(body.splitlines()) > 80:
        print("... (truncated)")

## 5. Document Extra Features and Metadata Handling
Scan the gateway class for optional features (capture/refund helpers, 3DS toggles, template overrides, order meta, debug switches) and surface code excerpts that indicate how they behave.

In [ ]:
def collect_feature_hits(source: str, token: str, window: int = 3):
    hits = []
    lowered = source.lower().splitlines()
    for idx, line in enumerate(lowered):
        if token in line:
            start = max(0, idx - window)
            end = min(len(lowered), idx + window + 1)
            hits.append((idx + 1, source.splitlines()[start:end]))
    return hits

feature_tokens = {
    "capture": "Manual capture / settlement",
    "refund": "Refund helper",
    "3d": "3-D Secure toggles",
    "template": "Custom template references",
    "meta": "Order meta writes",
    "debug": "Debug log switches",
    "nochex_apc": "Callback handling",
}

feature_rows = []
for token, label in feature_tokens.items():
    hits = collect_feature_hits(gateway_source.lower(), token)
    feature_rows.append({
        "keyword": token,
        "label": label,
        "occurrences": len(hits)
    })

feature_df = pd.DataFrame(feature_rows)
display(feature_df)

for token, label in feature_tokens.items():
    matches = collect_feature_hits(gateway_source, token)
    if not matches:
        continue
    print("\n" + "*" * 50)
    print(f"Keyword: {token} → {label}")
    for line_no, snippet in matches[:3]:
        print(f"Line {line_no}")
        print("\n".join(snippet))
        print("-")

## 6. Summarize Differences vs In-house Plugin
Blend the extracted Nochex facts with our CopyAndPay gateway behavior (Oppwa CopyAndPay/CopyAndPay API) to outline key differences across configuration, payment lifecycle, and auxiliary features.

In [ ]:
comparison_entries = [
    {
        "Area": "Credentials & Auth",
        "Nochex CopyPay": "TBD",
        "Our CopyAndPay": "TBD"
    }
]

comparison_df = pd.DataFrame(comparison_entries)
display(comparison_df)

comparison_md = comparison_df.to_markdown(index=False)
print(comparison_md)